In [1]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
import os

Configure

In [2]:
# folder with all rasters
raster_dir = "F:/tc_model_rasters"

In [3]:
# prediction depths
depths = [i for i in range(5,105,10)]

In [4]:
# folder containing model coefficients
model_folder = "C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch6_CASC_Project/Dual-Risk-Repo/NWCA_Model"
model_path = os.path.normpath(os.path.join(model_folder,'Best_TC_Concentration_Model.csv'))

In [5]:
# Output folder
output_dir = r"F:/tc_model_results"
os.makedirs(output_dir, exist_ok=True)

In [6]:
# Tile size — larger = faster but more RAM; 2048 is safe for most machines
TILE_SIZE = 2048

In [7]:
# NoData value for output
NODATA = -9999.0

In [8]:
# load best model
model_coeffs = pd.read_csv(model_path)

In [9]:
# set the intercept
intercept = model_coeffs.loc[0,"values"]

In [10]:
# get scalar depth effect
b_depth = model_coeffs.loc[1,"values"]

In [17]:
# list every raster file and its main-effect scalar
raster_coefficients = {}
for i in range(2,11):
    key = model_coeffs.loc[i,"labels"] + ".tif"
    raster_coefficients[key] = model_coeffs.loc[i,"values"]
print(raster_coefficients)

{'MAP.tif': np.float64(0.432018763535628), 'MAT.tif': np.float64(-48.0534813535329), 'DEM.tif': np.float64(-0.502968983337235), 'NDVI.tif': np.float64(802.785411316735), 'NDWI.tif': np.float64(2812.54307376681), 'Slope.tif': np.float64(-28.0904762224916), 'Aspect.tif': np.float64(-0.234022185146684), 'BiomassC_Stock.tif': np.float64(-0.2360816155979), 'SOC_Stock.tif': np.float64(0.896918926938801)}


In [13]:
# get coefficients for rasters that have interaction with depth
depth_raster_coefficients = {}
for i in range(11,14):
    key = model_coeffs.loc[i,"labels"].split(':')[1] + ".tif"
    depth_raster_coefficients [key] = model_coeffs.loc[i,"values"]

In [14]:
# list pairs and their interaction scalar
interaction_coefficients = {}
for i in range(14,36):
    keys = model_coeffs.loc[i,"labels"].split(':')
    for j in range(2):
        keys[j] = keys[j] + ".tif"
    keys = tuple(keys)
    interaction_coefficients[keys] = model_coeffs.loc[i,"values"]

Define helper functions

In [15]:
def full_path(filename):
    return os.path.normpath(os.path.join(raster_dir, filename))
 
def open_rasters(names):
    return {name: rasterio.open(full_path(name)) for name in names}
 
def read_tile(src, window):
    """Read a tile as float32, converting nodata to nan."""
    data = src.read(1, window=window).astype(np.float32)
    if src.nodata is not None:
        data[data == src.nodata] = np.nan
    return data

Check for raster alignment

In [18]:
files = list(raster_coefficients.keys())
ref = rasterio.open(full_path(files[0]))

print(f"Reference: {files[0]}")
print(f"  CRS:       {ref.crs}")
print(f"  Transform: {ref.transform}")
print(f"  Shape:     {ref.height} x {ref.width}")

all_aligned = True
for name in files[1:]:
    src = rasterio.open(full_path(name))
    aligned = (
        src.crs == ref.crs and
        src.transform == ref.transform and
        src.width == ref.width and
        src.height == ref.height
    )
    print(f"\n{name}: {'✓ aligned' if aligned else '✗ NOT aligned'}")
    if not aligned:
        print(f"  CRS:       {src.crs}")
        print(f"  Transform: {src.transform}")
        print(f"  Shape:     {src.height} x {src.width}")
    all_aligned = all_aligned and aligned

print(f"\n{'All rasters aligned!' if all_aligned else 'WARNING: misalignment detected — use WarpedVRT'}")

Reference: MAP.tif
  CRS:       EPSG:4326
  Transform: | 0.00, 0.00,-91.82|
| 0.00,-0.00, 42.91|
| 0.00, 0.00, 1.00|
  Shape:     202250 x 171912

MAT.tif: ✓ aligned


RasterioIOError: F:\tc_model_rasters\DEM.tif: No such file or directory

Main processing

In [16]:
# Open all rasters once — use first raster as reference for profile/shape
print("Opening rasters...")
all_raster_names = list(raster_coefficients.keys())
srcs = open_rasters(all_raster_names)
ref_name = all_raster_names[0]
ref = srcs[ref_name]
print ("Done")

Opening rasters...
Done


In [ ]:
profile = ref.profile.copy()
profile.update(dtype=rasterio.float32, count=1, nodata=NODATA, compress="lzw")
 
nrows, ncols = ref.height, ref.width
print(f"Raster shape: {nrows} rows x {ncols} cols ({nrows * ncols/1e6:.1f}M cells)")
 
for depth in depths:
    output_path = os.path.normpath(os.path.join(output_dir, f"tc_{depth}cm_concentration.tif"))
    print(f"\n{'='*50}")
    print(f"Processing depth {depth}cm -> {output_path}")
 
    with rasterio.open(output_path, "w", **profile) as dst:
        # Iterate over tiles
        for row_off in range(0, nrows, TILE_SIZE):
            row_count = min(TILE_SIZE, nrows - row_off)
            for col_off in range(0, ncols, TILE_SIZE):
                col_count = min(TILE_SIZE, ncols - col_off)
                window = Window(col_off, row_off, col_count, row_count)
 
                # Read all raster tiles
                tiles = {name: read_tile(srcs[name], window) for name in all_raster_names}
 
                # Initialize result with intercept
                result = np.full((row_count, col_count), intercept, dtype=np.float32)
 
                # Main effects: b_i * R_i
                #for name, coef in raster_coefficients.items():
                #    result += coef * tiles[name]
 
                # Depth scalar effect (modify as needed for your model)
                result += b_depth * depth
 
                # Interaction terms: b_ij * (R_i * R_j)
                #for (name_i, name_j), coef in interaction_coefficients.items():
                #    result += coef * (tiles[name_i] * tiles[name_j])
 
                # Depth * raster interactions
                #for name, coef in depth_raster_coefficients.items():
                #    result += coef * depth * tiles[name]
 
                # Restore nodata
                nan_mask = np.isnan(result)
                result[nan_mask] = NODATA
 
                dst.write(result, 1, window=window)
 
            print(f"  Row {row_off+row_count}/{nrows} complete...", end="\r")
 
    print(f"\n  Saved: {output_path}")
 
# Close all open rasters
for src in srcs.values():
    src.close()
 
print("\nAll depths complete.")

Raster shape: 202250 rows x 171912 cols (34769.2M cells)

Processing depth 5cm -> F:\tc_model_results\tc_5cm_concentration.tif
  Row 202250/202250 complete...
  Saved: F:\tc_model_results\tc_5cm_concentration.tif

Processing depth 15cm -> F:\tc_model_results\tc_15cm_concentration.tif
